# reduce-op-mean-divide — faded example 1: In-place divide for mean-reduction

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-op-mean-divide`. Running the beacon reports progress on the `Distributed: reduce-op mean divide` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce-op mean divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-op-mean-divide`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-op-mean-divide"
DD_SUBTOPIC = "Distributed: reduce-op mean divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Averaging across ranks is `all_reduce(SUM)` then an in-place divide by `world_size`. The divide must be in place (`/=`) so external references to the same storage continue to see the averaged result rather than being orphaned by a rebinding assignment.

## Faded exercise 1

Complete `mean_reduce`. The SUM all-reduce is already called. Fill in the in-place divide by `world_size` that turns the summed tensor into the mean while preserving the storage reference.

**Fill in:** divide the tensor in place by world_size to form the mean

In [ ]:
class FakeDist:
    def __init__(self, contribs):
        self.total = sum(contribs)
    def all_reduce(self, tensor, op='sum'):
        tensor.copy_(self.total)


def mean_reduce(tensor, world_size, dist_module):
    dist_module.all_reduce(tensor, op='sum')
    raise NotImplementedError()  # TODO: divide the tensor in place by world_size to form the mean


contribs = [t.tensor([float(r + 1)] * 2) for r in range(4)]
fd = FakeDist(contribs)
grad = t.tensor([1.0, 1.0])
ref = grad
mean_reduce(grad, 4, fd)

def _test():
    contribs = [t.tensor([float(r + 1)] * 2) for r in range(4)]  # 1+2+3+4 = 10
    fd = FakeDist(contribs)
    grad = t.tensor([1.0, 1.0])
    ref = grad
    mean_reduce(grad, 4, fd)
    # mean of [1,2,3,4] = 2.5 per element
    assert t.allclose(grad, t.tensor([2.5, 2.5])), f'expected 2.5, got {grad.tolist()}'
    assert grad is ref, 'in-place divide must preserve the storage reference'

try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class FakeDist:
    def __init__(self, contribs):
        self.total = sum(contribs)
    def all_reduce(self, tensor, op='sum'):
        tensor.copy_(self.total)


def mean_reduce(tensor, world_size, dist_module):
    dist_module.all_reduce(tensor, op='sum')
    tensor /= world_size


contribs = [t.tensor([float(r + 1)] * 2) for r in range(4)]
fd = FakeDist(contribs)
grad = t.tensor([1.0, 1.0])
ref = grad
mean_reduce(grad, 4, fd)
```
</details>